# Week 37, Day 5: Defensive programming

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w037d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def find(names, wanted):
    for i, name in enumerate(names):
        if name == wanted:
            return i
    return -1

team = ["Ada", "Bo", "Chi"]
captain = team[find(team, "Bo")]
vice = team[find(team, "Dee")]
print(captain, vice)

In [ ]:
def withdraw(balance, amount):
    """Take amount from balance.

    Needs: amount is a positive whole number, no more than balance.
    Promises: returns the new balance, which is not negative.
    """
    if not isinstance(amount, int) or amount <= 0:
        raise ValueError(f"amount must be a positive whole number, not {amount!r}")
    if amount > balance:
        raise ValueError(f"can't take {amount} from {balance}")
    new = balance - amount
    assert new >= 0, "postcondition broken"
    return new

print(withdraw(100, 30))
for amount in [0, 500, 2.5]:
    try:
        withdraw(100, amount)
    except ValueError as err:
        print("refused:", err)

In [ ]:
class Queue:
    """People waiting, in order, each at most once."""

    def __init__(self):
        self._people = []
        self._check()

    def join(self, person):
        if person in self._people:
            raise ValueError(f"{person} is already waiting")
        self._people.append(person)
        self._check()

    def serve(self):
        if not self._people:
            raise LookupError("nobody is waiting")
        person = self._people.pop(0)
        self._check()
        return person

    def _check(self):
        assert len(set(self._people)) == len(self._people), f"someone is in the queue twice: {self._people}"

q = Queue()
q.join("Ada")
q.join("Bo")
print(q.serve(), q._people)

In [ ]:
def find(names, wanted):
    try:
        return names.index(wanted)
    except ValueError:
        raise LookupError(f"{wanted!r} isn't in the team") from None

team = ["Ada", "Bo", "Chi"]
print(team[find(team, "Bo")])
try:
    print(team[find(team, "Dee")])
except LookupError as err:
    print("stopped:", err)

In [ ]:
class Team:
    def __init__(self, names):
        if len(names) > 5:
            raise ValueError("at most 5 players")
        self._names = list(names)            # a copy: the caller's list stays theirs

    @property
    def names(self):
        return tuple(self._names)            # a read-only view for callers

    def add(self, name):
        if len(self._names) >= 5:
            raise ValueError("the team is full")
        self._names.append(name)

picked = ["Ada", "Bo"]
team = Team(picked)
picked.extend(["Chi", "Dee", "Ed", "Fay", "Gus"])    # the caller keeps using their list
print(team.names, len(team.names))
try:
    team.names.append("Hal")
except AttributeError as err:
    print("can't change it:", err)

In [ ]:
def parse_answer(text):
    cleaned = text.strip().lower()
    if cleaned in ("y", "yes", "true", "1"):
        return True
    if cleaned in ("n", "no", "false", "0"):
        return False
    raise ValueError(f"please answer yes or no, not {text!r}")

for text in ["Yes", "  n ", "TRUE", "maybe"]:
    try:
        print(repr(text), "->", parse_answer(text))
    except ValueError as err:
        print(err)

> **Key idea.** A contract states a function's preconditions (the caller's side), postconditions (the function's promise) and a class's invariants. Check preconditions and anything from outside with `raise`; use `assert` only for things that can fail through your own bug (postconditions, invariants), because `python -O` removes asserts. Never put side effects in an assert. Fail fast: raise instead of returning special values like `-1`, and check settings at start-up. Copy mutable data you store from outside, and return tuples or other read-only views. Accept variations at the boundary, normalise them once, reject what's unclear, and be strict inside.

### Exercise: Checking the contract

Write `percent(part, whole)`, which returns `part` as a percentage of `whole`, rounded to 1 decimal place. Its preconditions: both are numbers (`int` or `float`, but not `bool`), otherwise `TypeError`; `whole` is above 0 and `part` is from 0 to `whole`, otherwise `ValueError`. Check them with `raise`, not `assert`.

In [ ]:
def percent(part, whole):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_percent():\n    \"the contract\"\n    assert percent(1, 3) == 33.3 and percent(0, 5) == 0.0 and percent(2.5, 10) == 25.0 and percent(7, 7) == 100.0\n    for args, error in [((1, 0), ValueError), ((-1, 5), ValueError), ((6, 5), ValueError), ((\"1\", 5), TypeError), ((1, True), TypeError), ((None, 2), TypeError)]:\n        try:\n            percent(*args)\n            assert False, f\"percent{args} should raise {error.__name__}\"\n        except error:\n            pass\n    assert \"assert\" not in source, \"preconditions need raise: asserts can be switched off\"")

### Exercise: A playlist that keeps its rules

Write a `Playlist` class whose rule is "no song appears twice, and there are at most `limit` songs". `Playlist(songs, limit=10)` checks the songs (raise `ValueError` for a duplicate or too many) and stores a **copy**. `add(song)` raises `ValueError` for a duplicate or when full. The `songs` property returns a **tuple**. Code outside mustn't be able to break the rule by changing the list it passed in, or the result of `songs`.

In [ ]:
class Playlist:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_playlist():\n    \"the rules\"\n    mine = [\"a\", \"b\"]\n    p = Playlist(mine, limit=3)\n    mine.extend([\"a\", \"c\", \"d\"])\n    assert p.songs == (\"a\", \"b\"), \"store a copy\"\n    p.add(\"c\")\n    assert p.songs == (\"a\", \"b\", \"c\") and isinstance(p.songs, tuple)\n    for song in [\"d\", \"a\"]:\n        try:\n            p.add(song)\n            assert False, f\"adding {song!r} should raise\"\n        except ValueError:\n            pass\n    for songs in [[\"x\", \"x\"], [\"1\", \"2\", \"3\", \"4\"]]:\n        try:\n            Playlist(songs, limit=3)\n            assert False, f\"{songs} should be refused\"\n        except ValueError:\n            pass\n    q = Playlist([])\n    q.add(\"z\")\n    assert q.songs == (\"z\",)")

### Exercise: Generous at the edge

Write `parse_minutes(text)`, which turns a duration into a number of minutes. Accept `"90m"`, `"2h"` and `"1h30m"`, in any case, with spaces anywhere (`" 1H 30M "`). Reject, with `ValueError`, anything else: an empty string, a missing number (`"h"`), a decimal (`"1.5h"`), minutes of 60 or more after hours (`"1h75m"`), or anything left over (`"2h5"`). Normalise first (remove spaces, lower case), then be strict.

In [ ]:
def parse_minutes(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_good():\n    \"accepted\"\n    for text, want in [(\"90m\", 90), (\"2h\", 120), (\"1h30m\", 90), (\" 1H 30M \", 90), (\"0m\", 0), (\"10H\", 600), (\"1h0m\", 60)]:\n        assert parse_minutes(text) == want, f\"{text!r}: got {parse_minutes(text)}\"\ndef test_bad():\n    \"rejected\"\n    for text in [\"\", \"  \", \"h\", \"m\", \"1.5h\", \"1h75m\", \"2h5\", \"abc\", \"5\", \"1m2h\", \"-3m\", \"1hh\"]:\n        try:\n            parse_minutes(text)\n            assert False, f\"{text!r} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: An invariant on guard

Write `SortedScores`, which keeps a list of scores always in ascending order. `add(score)` puts the score in the right place (use `bisect.insort` from Week 18, or any way you like); `top(n)` returns the `n` highest, highest first, as a list. Write a `_check` method that **asserts** the list is sorted, and call it at the end of `add`. The checks will deliberately corrupt the private list to make sure your `_check` catches it.

In [ ]:
import bisect

class SortedScores:
    def __init__(self):
        self._scores = []
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_scores():\n    \"scores\"\n    s = SortedScores()\n    for x in [50, 20, 90, 70, 20]:\n        s.add(x)\n    assert s._scores == [20, 20, 50, 70, 90] and s.top(2) == [90, 70] and s.top(0) == [] and s.top(10) == [90, 70, 50, 20, 20]\ndef test_guard():\n    \"the invariant\"\n    s = SortedScores()\n    s.add(5)\n    s._scores.append(1)          # someone breaks the rule from outside\n    try:\n        s.add(10)\n        assert False, \"_check should catch the unsorted list\"\n    except AssertionError:\n        pass")

### Bug hunt: A check that can vanish

`set_volume` uses `assert` to refuse a bad level from the caller. That check would disappear under `python -O`, and even now it raises the wrong kind of error, so the caller's `except ValueError` doesn't catch it. Fix it so the output is `volume 7` then `refused: volume must be from 0 to 10, not 15`.

In [ ]:
def set_volume(level):
    assert 0 <= level <= 10, f"volume must be from 0 to 10, not {level}"
    return f"volume {level}"

for level in [7, 15]:
    try:
        print(set_volume(level))
    except ValueError as err:
        print("refused:", err)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"raise, not assert\"\n    expect_output(\"volume 7\\nrefused: volume must be from 0 to 10, not 15\")\n    assert \"assert\" not in source")

### Bug hunt: The list that changed by itself

`Course` checks that it has at most 3 students, but it keeps the caller's list itself, so when the caller reuses the list for another course, this course grows to 5. Fix `Course` so the output is `('Ada', 'Bo')`, then `too many students` (the five names are refused for a new course).

In [ ]:
class Course:
    def __init__(self, students):
        if len(students) > 3:
            raise ValueError("too many students")
        self.students = students

names = ["Ada", "Bo"]
maths = Course(names)
names += ["Chi", "Dee", "Ed"]
print(tuple(maths.students))
try:
    Course(names)
except ValueError as err:
    print(err)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a copy\"\n    expect_output(\"('Ada', 'Bo')\\ntoo many students\")")

### Bug hunt: Failing slowly

`find_price` returns `0` when an item isn't sold, so the missing item quietly makes the basket cheaper. Make it fail fast: raise `LookupError(f"no price for {item!r}")`, so the output is `total 450` then `stopped: no price for 'jam'`.

In [ ]:
PRICES = {"tea": 300, "milk": 150}

def find_price(item):
    return PRICES.get(item, 0)

def total(basket):
    return sum(find_price(item) for item in basket)

for basket in [["tea", "milk"], ["tea", "jam"]]:
    try:
        print("total", total(basket))
    except LookupError as err:
        print("stopped:", err)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fail fast\"\n    expect_output(\"total 450\\nstopped: no price for 'jam'\")")

### Puzzle: Contract first

Arrange the lines into a function that checks its precondition with `raise`, does its work, and asserts its postcondition. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
def f(items):
    stored = list(items)
    return stored

original = [1, 2]
copy = f(original)
copy.append(3)

a = original
b = __debug__
c = (1, 2) + (3,)
d = "  Yes ".strip().lower() in ("y", "yes")
e = isinstance(True, int)

p = ___   # a
q = ___   # b
r = ___   # c
s = ___   # d
t = ___   # e

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2] and q is True and r == (1, 2, 3), \"p, q and r\"\n    assert s is True and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a room diary

Write a `Diary` for one meeting room. Bookings are `(start, end, who)`, with times as minutes since midnight. Rules (the invariant): bookings are kept sorted by start time and never overlap (one ending at 600 and another starting at 600 is fine).

- `book(start, end, who)` accepts the times as **text** like `"9:30"` or `" 09:30 "` (normalise: strip spaces; hours 0 to 23, minutes 00 to 59, always two digits for minutes). Bad times raise `ValueError`. `end` must be after `start`, and `who` must be non-empty after stripping (store it stripped), else `ValueError`. A clash raises `ValueError` naming who has the room: `"clashes with Ada"`. Nothing changes on any error.
- `bookings` is a property returning a **tuple** of the `(start, end, who)` tuples.
- `_check()` asserts the invariant, and `book` calls it after every change.

In [ ]:
class Diary:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_book():\n    \"booking\"\n    d = Diary()\n    d.book(\"9:30\", \"10:00\", \"Ada\")\n    d.book(\" 08:00 \", \"9:30\", \"  Bo \")\n    d.book(\"10:00\", \"11:15\", \"Chi\")\n    assert d.bookings == ((480, 570, \"Bo\"), (570, 600, \"Ada\"), (600, 675, \"Chi\")), d.bookings\n    assert isinstance(d.bookings, tuple)\ndef test_refused():\n    \"refusals\"\n    d = Diary()\n    d.book(\"9:30\", \"10:00\", \"Ada\")\n    for args, words in [((\"9:45\", \"10:30\", \"Bo\"), \"clashes with Ada\"), ((\"9:00\", \"9:31\", \"Bo\"), \"clashes with Ada\"), ((\"9:00\", \"11:00\", \"Bo\"), \"clashes with Ada\"),\n                        ((\"11:00\", \"10:00\", \"Bo\"), \"\"), ((\"24:00\", \"24:30\", \"Bo\"), \"\"), ((\"9:5\", \"10:00\", \"Bo\"), \"\"), ((\"11:00\", \"12:00\", \"  \"), \"\"), ((\"noon\", \"13:00\", \"Bo\"), \"\")]:\n        try:\n            d.book(*args)\n            assert False, f\"{args} should be refused\"\n        except ValueError as err:\n            assert words in str(err), f\"{args}: message {str(err)!r}\"\n    assert d.bookings == ((570, 600, \"Ada\"),), \"nothing should change\"\ndef test_check():\n    \"the invariant\"\n    d = Diary()\n    d.book(\"9:00\", \"10:00\", \"Ada\")\n    d._bookings.append((500, 560, \"X\"))     # corrupted from outside: overlapping and out of order\n    try:\n        d.book(\"12:00\", \"13:00\", \"Bo\")\n        assert False, \"_check should catch it\"\n    except AssertionError:\n        pass")

## Warm-up

There's no Dee in the team, yet the program happily chose Chi as vice-captain. `find` reported "not found" with `-1`, which is also a perfectly good list index, so the mistake **travelled on** silently. Nothing crashed, and the answer was wrong. Today is about making programs stop at the first sign of trouble, as close to the cause as possible.

## Contracts

Bertrand Meyer's idea of **design by contract** (from his book *Object-Oriented Software Construction*, and the Eiffel language) gives a clear way to think about who checks what. Every function has a contract with its callers:

| Part | Meaning | Whose job |
|---|---|---|
| **Precondition** | what must be true when the function is called | the caller's, to meet; the function's, to check |
| **Postcondition** | what the function promises when it returns | the function's |
| **Invariant** | what is always true of an object between method calls | the class's |

- The **preconditions** are checked with `if` and `raise`, because a caller can break them: they're part of the function's public behaviour, written in its docstring.
- The **postcondition** is checked with `assert`, because if it fails, the function itself has a bug. No caller can cause it.

Classes have **invariants** (Week 28): rules true for every instance, between method calls. A handy habit for a class with tricky rules is a `_check` method that asserts them all, called at the end of each method that changes the object:

If a later change to `join` forgot the duplicate check, `_check` would catch it the first time it mattered, in the method that caused it, instead of three screens later when someone is served twice.

## assert or raise?

Week 7 introduced `assert`, and warned not to use it on what a user typed. Here's the full rule:

| Use `raise` when | Use `assert` when |
|---|---|
| the problem can come from **outside** the code: a caller, a user, a file, the network | the problem can only mean **your own code** has a bug |
| it's part of the function's promised behaviour | it's a note to yourself: "this must be true here" |
| e.g. a negative amount passed to `withdraw` | e.g. a postcondition, an invariant, "this list is sorted by now" |

The reason is that **asserts can be switched off**. Running `python -O program.py` (optimise) removes every `assert` statement, and sets the built-in `__debug__` to `False`. So for this file:

```python
print(__debug__)
assert 1 > 2, "never"
print("still running")
```

```text
$ python check.py
True
Traceback (most recent call last):
  File "check.py", line 2, in <module>
    assert 1 > 2, "never"
           ^^^^^
AssertionError: never

$ python -O check.py
False
still running
```

Any check that must always happen, such as refusing a negative amount, can't be an `assert`. Two more rules follow:

- **No side effects in an assert.** `assert queue.pop() == "end"` pops only when asserts are on, so the program behaves differently with `-O`.
- **Don't catch `AssertionError`** to handle it: a failed assert means a bug, and the fix is in the code.

Very few programs are run with `-O`, but the rule holds anyway, because it tells readers what each check is for: an `assert` says "if this fails, I made a mistake"; a `raise` says "if this fails, you gave me something wrong".

## Fail fast

The warm-up's `-1` is an example of a program **failing slowly**: something went wrong, but the program carried on, and the damage showed up later and elsewhere. Failing **fast** means stopping at the first sign of trouble, where the cause is easiest to see:

| Failing slowly | Failing fast |
|---|---|
| return `-1`, `None` or `""` for "it didn't work", and hope the caller checks | raise an exception the caller can't ignore |
| skip a bad record silently | raise, or at least log a warning (Day 4) and count it |
| fill in a made-up default for a missing setting | refuse to start, naming the setting |
| `except Exception: pass` | catch only what you can handle (Day 1) |

Returning `None` for "not found" is fine when the caller is **expected** to check (like `dict.get`), and the name makes that clear. The danger is a special value that looks like a real answer, like `-1`, `0` or `""`.

Failing fast is also why programs should **check settings at start-up** (Week 36): a typo in a setting should stop the program in the first second, not when the first customer pays.

## Defensive copies

An object can't keep its invariants if code outside can change its data behind its back. Lists, dictionaries and sets are passed around by reference (Week 9's aliasing), so storing or returning one shares it:

Without `list(names)`, the caller's later `extend` would have given the team seven players, breaking the rule the constructor checked. Copy what you **store** if it came from outside, and hand out copies or read-only versions (a tuple, a `frozenset`, `types.MappingProxyType` for a dictionary) of what you **return**. Immutable objects (frozen dataclasses from Week 32, tuples, strings) need no copies at all, which is one of the best arguments for using them.

Copies cost time and memory, so don't copy everything everywhere: copy at the edges of an object that has rules to keep.

## Generous at the edge, strict inside

An old rule from the design of the internet, often called Postel's law after Jon Postel, who wrote it into the TCP standard (RFC 761, 1980), says: "be conservative in what you do, be liberal in what you accept from others."

For a program's input, a careful version of this works well: at the **boundary**, accept reasonable variations (spaces, capital letters, `"yes"` or `"y"`), **normalise** them once into one exact form, and **reject** anything genuinely unclear. Inside, every function can then be strict and simple:

Being generous has limits. Guessing what unclear input meant ("maybe" means yes?) creates bugs that are hard to find, and more recent advice from the people who design internet standards (RFC 9413, *Maintaining Robust Protocols*, 2023) warns that tolerating broken input forever makes everyone depend on the tolerance. So: accept **variations** of valid input, never **guesses** about invalid input.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a function of yours that returns `-1`, `None` or `""` to mean "it didn't work". Would its callers notice if they forgot to check? Tomorrow is the Forge: a robust importer that uses everything from this week.